# PET-PINN Adaptation — 1D Counter-Flow Heat Exchanger
**Applied ML for Scientists & Engineers — Final Project (Ehud Ischakbaev & Daniel Azouri)**

**Anchor paper:** Fu, W., Wang, H., and Wang, J., *"Efficient adaptation of physics-informed neural networks for parametric thermal convection using adapter layers,"* Appl. Phys. Lett. **128**, 083901 (2026).

We reproduce the paper's central comparison — **adapter-based fine-tuning (PET-PINN) vs. full fine-tuning** — on a reduced, fully controlled system: the 1D counter-flow heat exchanger, whose exact analytical solution provides ground truth at **every** operating point $(k_h, k_c)$.

$$\frac{dT_h}{dx} = -k_h(T_h - T_c), \qquad \frac{dT_c}{dx} = -k_c(T_h - T_c), \qquad T_h(0)=100\,^\circ\mathrm{C},\; T_c(L)=20\,^\circ\mathrm{C}$$

**Experiment (mirrors Table I of Fu et al.):**
1. **Pre-train** a baseline PINN at the source point $(k_h,k_c)=(1.0,1.0)$.
2. **Adapt** to the target $(2.0, 0.8)$ three ways: (a) from scratch [reference], (b) full fine-tuning, (c) frozen backbone + rank-$r$ adapters.
3. **Compare:** relative $L_2$ error vs. the closed-form solution, trainable-parameter count, wall-clock time.

Note: implemented in **pure PyTorch** (not DeepXDE) — selective layer freezing and adapter injection require direct control of the module graph.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# 0. Reproducibility & problem constants
# ----------------------------------------------------------------------------
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float64
torch.set_default_dtype(DTYPE)

L = 1.0
T_H_IN, T_C_IN = 100.0, 20.0
T_REF, T_SPAN = T_C_IN, (T_H_IN - T_C_IN)

SOURCE = (1.0, 1.0)      # pre-training operating point
TARGET = (2.0, 0.8)      # adaptation target

N_COL = 100              # collocation points
PRETRAIN_ADAM = 10000
ADAPT_ADAM = 3000        # same Adam budget for full-FT and adapters
ADAM_LR = 1e-3
ADAPTER_RANK = 4         # r=4 as in Fu et al.

## Analytical ground truth
$\Delta T = T_h - T_c$ obeys $\Delta T(x) = D_0 e^{-\lambda x}$ with $\lambda = k_h - k_c$; $D_0$ follows from the boundary conditions. Valid for **any** $(k_h,k_c)$ — exact reference at every adaptation target.

In [ ]:
def analytical_solution(x, k_h, k_c):
    """Exact solution with T_h(0)=T_H_IN, T_c(L)=T_C_IN."""
    x = np.asarray(x, dtype=float)
    lam = k_h - k_c
    dT_in = T_H_IN - T_C_IN
    if np.isclose(lam, 0.0):
        D0 = dT_in / (1.0 + k_h * L)
        T_h = T_H_IN - k_h * D0 * x
        dT = D0 * np.ones_like(x)
    else:
        denom = k_h * (1.0 - np.exp(-lam * L)) / lam + np.exp(-lam * L)
        D0 = dT_in / denom
        T_h = T_H_IN - k_h * D0 * (1.0 - np.exp(-lam * x)) / lam
        dT = D0 * np.exp(-lam * x)
    return T_h, T_h - dT


def relative_l2(pred, exact):
    return np.linalg.norm(pred - exact) / np.linalg.norm(exact)


# ----------------------------------------------------------------------------
# 1. Models

## Models

- **`BasePINN`** — same architecture as our verified DeepXDE baseline: $[1] + 3\times[20] + [2]$, tanh, output transform to the physical temperature range.
- **`Adapter`** — bottleneck module, Eq. (2) of Fu et al.: $h' = h + W_{down}\,\sigma(W_{up} h)$ with rank $r=4$ (like LoRA). $W_{down}$ initialized to **zero**, so the adapted network starts exactly identical to the frozen backbone.
- **`AdaptedPINN`** — frozen backbone + one adapter after each hidden activation (Eq. (3) of Fu et al.). Only the adapters receive gradients.

In [ ]:
class BasePINN(nn.Module):
    """FNN [1] + 3x[20] + [2], tanh, matching the DeepXDE baseline.
    Output transform rescales O(1) outputs to the physical range."""

    def __init__(self, width=20, depth=3):
        super().__init__()
        self.inp = nn.Linear(1, width)
        self.hidden = nn.ModuleList(
            [nn.Linear(width, width) for _ in range(depth - 1)])
        self.out = nn.Linear(width, 2)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def hidden_states(self, x):
        """Returns the post-activation hidden states h1..h_depth."""
        hs = [torch.tanh(self.inp(x))]
        for lin in self.hidden:
            hs.append(torch.tanh(lin(hs[-1])))
        return hs

    def forward(self, x):
        h = self.hidden_states(x)[-1]
        return T_REF + T_SPAN * self.out(h)


class Adapter(nn.Module):
    """Bottleneck adapter, Eq. (2) of Fu et al.:
    h' = h + W_down * sigma(W_up * h), r << d.
    W_down initialized to zero so the adapted network starts exactly
    equal to the frozen backbone (identity residual)."""

    def __init__(self, d=20, r=ADAPTER_RANK):
        super().__init__()
        self.up = nn.Linear(d, r)
        self.down = nn.Linear(r, d)
        nn.init.xavier_normal_(self.up.weight)
        nn.init.zeros_(self.up.bias)
        nn.init.zeros_(self.down.weight)   # identity start
        nn.init.zeros_(self.down.bias)
        self.act = nn.GELU()

    def forward(self, h):
        return h + self.down(self.act(self.up(h)))


class AdaptedPINN(nn.Module):
    """Frozen BasePINN backbone with one adapter after each hidden
    activation (Eq. (3) of Fu et al.). Only adapters are trainable."""

    def __init__(self, base: BasePINN, r=ADAPTER_RANK):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        d = base.inp.out_features
        n_hidden = 1 + len(base.hidden)
        self.adapters = nn.ModuleList([Adapter(d, r) for _ in range(n_hidden)])

    def forward(self, x):
        h = self.adapters[0](torch.tanh(self.base.inp(x)))
        for lin, ad in zip(self.base.hidden, self.adapters[1:]):
            h = ad(torch.tanh(lin(h)))
        return T_REF + T_SPAN * self.base.out(h)


def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Physics loss and training loop
Composite loss = mean squared ODE residuals at 100 collocation points + Dirichlet BC errors at the two inlets. Two-stage optimization: Adam followed by L-BFGS, applied **only to the trainable parameters** (crucial for the adapter case).

In [ ]:
X_COL = torch.linspace(0.0, L, N_COL, device=DEVICE).reshape(-1, 1)
X0 = torch.zeros(1, 1, device=DEVICE)   # hot inlet
XL = torch.full((1, 1), L, device=DEVICE)  # cold inlet


def pinn_loss(model, k_h, k_c):
    x = X_COL.clone().requires_grad_(True)
    out = model(x)
    T_h, T_c = out[:, 0:1], out[:, 1:2]
    dTh = torch.autograd.grad(T_h, x, torch.ones_like(T_h), create_graph=True)[0]
    dTc = torch.autograd.grad(T_c, x, torch.ones_like(T_c), create_graph=True)[0]
    res_h = dTh + k_h * (T_h - T_c)
    res_c = dTc + k_c * (T_h - T_c)
    bc = ((model(X0)[:, 0] - T_H_IN) ** 2).mean() + \
         ((model(XL)[:, 1] - T_C_IN) ** 2).mean()
    return (res_h ** 2).mean() + (res_c ** 2).mean() + bc


def train(model, k_h, k_c, adam_iters, tag=""):
    """Adam then L-BFGS on the trainable parameters only.
    Returns (loss_history, wall_clock_seconds)."""
    params = [p for p in model.parameters() if p.requires_grad]
    t0 = time.time()
    history = []

    opt = torch.optim.Adam(params, lr=ADAM_LR)
    for it in range(adam_iters):
        opt.zero_grad()
        loss = pinn_loss(model, k_h, k_c)
        loss.backward()
        opt.step()
        if it % 100 == 0 or it == adam_iters - 1:
            history.append((it, loss.item()))

    lbfgs = torch.optim.LBFGS(params, max_iter=500, history_size=50,
                              tolerance_grad=1e-12, tolerance_change=1e-14,
                              line_search_fn="strong_wolfe")

    def closure():
        lbfgs.zero_grad()
        loss = pinn_loss(model, k_h, k_c)
        loss.backward()
        return loss

    lbfgs.step(closure)
    wall = time.time() - t0
    final = pinn_loss(model, k_h, k_c).item()
    history.append((adam_iters, final))
    print(f"[{tag}] final loss {final:.3e}   wall {wall:.1f}s   "
          f"trainable params {n_trainable(model)}")
    return history, wall


def evaluate(model, k_h, k_c, n=200):
    x = np.linspace(0.0, L, n).reshape(-1, 1)
    with torch.no_grad():
        pred = model(torch.tensor(x, device=DEVICE)).cpu().numpy()
    Th_e, Tc_e = analytical_solution(x.ravel(), k_h, k_c)
    return (relative_l2(pred[:, 0], Th_e), relative_l2(pred[:, 1], Tc_e),
            x, pred, Th_e, Tc_e)

## Run the full experiment

In [ ]:
def main():
    results = []

    # --- Stage 1: pre-train baseline at SOURCE ---
    print(f"\n=== Pre-training at source (k_h,k_c)={SOURCE} ===")
    base = BasePINN().to(DEVICE)
    _, wall_pre = train(base, *SOURCE, PRETRAIN_ADAM, tag="pretrain")
    eh, ec, *_ = evaluate(base, *SOURCE)
    print(f"    source rel L2: T_h {eh:.2e}, T_c {ec:.2e}")

    # sanity: pre-trained model evaluated at TARGET (no adaptation)
    eh0, ec0, *_ = evaluate(base, *TARGET)
    print(f"    UNADAPTED error at target: T_h {eh0:.2e}, T_c {ec0:.2e}")
    results.append(("No adaptation", 0, 0.0, eh0, ec0))

    # --- Method A: from scratch at TARGET (reference) ---
    print(f"\n=== (a) From scratch at target {TARGET} ===")
    torch.manual_seed(SEED)
    scratch = BasePINN().to(DEVICE)
    _, wall_a = train(scratch, *TARGET, PRETRAIN_ADAM, tag="scratch")
    eh_a, ec_a, *_ = evaluate(scratch, *TARGET)
    results.append(("From scratch", n_trainable(scratch), wall_a, eh_a, ec_a))

    # --- Method B: full fine-tuning ---
    print(f"\n=== (b) Full fine-tuning ({ADAPT_ADAM} Adam iters) ===")
    import copy
    full_ft = copy.deepcopy(base)
    for p in full_ft.parameters():
        p.requires_grad_(True)
    _, wall_b = train(full_ft, *TARGET, ADAPT_ADAM, tag="full-FT")
    eh_b, ec_b, *_ = evaluate(full_ft, *TARGET)
    results.append(("Full fine-tuning", n_trainable(full_ft), wall_b, eh_b, ec_b))

    # --- Method C: adapters (PET-PINN) ---
    print(f"\n=== (c) Adapters, rank r={ADAPTER_RANK} ({ADAPT_ADAM} Adam iters) ===")
    pet = AdaptedPINN(copy.deepcopy(base)).to(DEVICE)
    _, wall_c = train(pet, *TARGET, ADAPT_ADAM, tag="adapters")
    eh_c, ec_c, x, pred, Th_e, Tc_e = evaluate(pet, *TARGET)
    results.append((f"Adapters (r={ADAPTER_RANK})", n_trainable(pet), wall_c, eh_c, ec_c))

    # --- Table I analogue ---
    print("\n" + "=" * 78)
    print(f"TABLE (source {SOURCE} -> target {TARGET}, seed {SEED}, {DEVICE})")
    print("=" * 78)
    print(f"{'Method':<20}{'Trainable':>12}{'Wall (s)':>10}"
          f"{'rel L2 T_h':>14}{'rel L2 T_c':>14}")
    for name, npar, wall, eh_, ec_ in results:
        print(f"{name:<20}{npar:>12}{wall:>10.1f}{eh_:>14.2e}{ec_:>14.2e}")

    # --- Figure: adapter prediction vs analytical at target ---
    plt.figure(figsize=(8, 5))
    plt.plot(x, pred[:, 0], "r-", lw=2.5, label="Hot fluid $T_h$ (PET-PINN)")
    plt.plot(x, pred[:, 1], "b-", lw=2.5, label="Cold fluid $T_c$ (PET-PINN)")
    plt.plot(x, Th_e, "k--", lw=1.5, label="Analytical (exact)")
    plt.plot(x, Tc_e, "k--", lw=1.5)
    plt.xlabel("Position along the exchanger $x$, m", fontsize=12)
    plt.ylabel("Temperature, $^\\circ$C", fontsize=12)
    plt.title(f"Adapter-based transfer to target $(k_h,k_c)$={TARGET}; "
              f"rel. $L_2$: $T_h$ {eh_c:.1e}, $T_c$ {ec_c:.1e}", fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, ls="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig("profiles_adapter_target.png", dpi=300)
    plt.close()
    print("\nSaved: profiles_adapter_target.png")
    return results

results = main()

## Notes for the report

- The **"No adaptation"** row is the key motivation: the pre-trained model evaluated directly at the target has ~30% error — adaptation is genuinely needed.
- All three adaptation methods reach $L_2 \sim 10^{-6}$; the interesting comparison is **cost**: trainable parameters (552 vs. 922) and wall-clock.
- On this small network the adapter advantage is modest — an honest finding worth discussing (the paper's 8×20 network shows a larger gap: 1280 vs. 3063 params). Sweeping adapter rank $r$ and multiple targets across the $(k_h,k_c)$ plane is the planned extension.
- Seed is fixed (2026); results are reproducible on the same software/hardware stack.

In [ ]:
from google.colab import files
files.download("profiles_adapter_target.png")